# Campus Energy Consumption 24-Hour Forecasting & Peak Advisory
**Objective:** Develop an AI model to forecast campus electrical load for the next 24 hours using recorded hourly time-series data.

### Challenge Requirements Addressed:
1. **Original Historical Dataset:** Ingests recorded hourly electricity consumption, ambient temperature, day of week, and occupancy status.
2. **Chronological Splitting:** Enforces strict forward-in-time partitioning (Train $\rightarrow$ Val $\rightarrow$ Test) without mixing future data into training.
3. **Baseline Benchmarking:** Compares against the building manager's standard persistence heuristic ( = y_{t-24}$) and weekly persistence ( = y_{t-168}$).
4. **Peak Demand Flagging:** Automatically flags projected high-consumption periods and recommends practical operational facility checks (pre-cooling, static pressure resets) without automated equipment control.
5. **Data Limitations:** Documents real-world constraints: weather uncertainty, thermal inertia lag, and sensor calibration limits.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE_DIR = Path('..').resolve()
sys.path.append(str(BASE_DIR))

from src.config import DATA_PATH, PEAK_PERCENTILE
from src.data_loader import load_dataset, split_chronological, prepare_data
from src.models import (
    PersistenceBaseline24h,
    PersistenceBaseline168h,
    RidgeEnergyForecaster,
    HistGradientBoostingForecaster,
    XGBoostEnergyForecaster,
)
from src.evaluator import evaluate_models_comparison, calculate_metrics
from src.advisory import CampusAdvisoryEngine

print('Libraries imported successfully.')

## 1. Load Provided Campus Energy Dataset
We load the recorded hourly telemetry dataset containing 13,128 consecutive measurements.

In [2]:
df = pd.read_csv(DATA_PATH)
df["timestamp"] = pd.to_datetime(df["timestamp"])
print(f"Dataset Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head(10)

In [3]:
# Descriptive Statistics of Recorded Measurements
df.describe().T

## 2. Exploratory Analysis: Load Profiles & Temperature Correlation
We inspect how electrical demand fluctuates across the day (diurnal cycle) and how temperature drives cooling energy.

In [4]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.5))

# Average hourly profile by weekday vs weekend
wd_hourly = df[df["is_weekend"] == 0].groupby("hour")["electricity_consumption_kwh"].mean()
we_hourly = df[df["is_weekend"] == 1].groupby("hour")["electricity_consumption_kwh"].mean()

ax1.plot(wd_hourly.index, wd_hourly.values, label="Weekday (Instructional)", color="#0b60b0", lw=2.5)
ax1.plot(we_hourly.index, we_hourly.values, label="Weekend (Unoccupied)", color="#7f7f7f", lw=2.0, ls="--")
ax1.set_title("Diurnal Campus Load Curve (kWh)", fontweight="bold")
ax1.set_xlabel("Hour of Day (00:00 - 23:00)")
ax1.set_ylabel("Mean Electricity Demand (kWh)")
ax1.grid(True, alpha=0.3)
ax1.legend()

# Temperature vs Electrical Demand
sample = df.sample(min(1500, len(df)), random_state=42)
sc = ax2.scatter(sample["temperature_c"], sample["electricity_consumption_kwh"], c=sample["is_occupied"], cmap="coolwarm", alpha=0.5, s=20)
ax2.axvline(18.0, color="orange", ls=":", label="Cooling Base (18°C)")
ax2.set_title("Outdoor Temperature vs. Electricity Demand", fontweight="bold")
ax2.set_xlabel("Ambient Temperature (°C)")
ax2.set_ylabel("Electricity Consumption (kWh)")
ax2.grid(True, alpha=0.3)
ax2.legend()

plt.tight_layout()
plt.show()

## 3. Strict Chronological Partitioning
In time-series forecasting, standard random shuffling () creates lookahead leakage by placing future hours into training.

We enforce strict chronological order:
- **Train Set (70%):** 2024-01-08 to 2025-01-19 (9,072 hours)
- **Validation Set (15%):** 2025-01-20 to 2025-04-10 (1,944 hours)
- **Holdout Test Set (15%):** 2025-04-11 to 2025-06-30 (1,944 hours)

In [5]:
train_df, val_df, test_df, feature_cols, target_col = prepare_data(DATA_PATH)

print("Chronological Partition Verification:")
print(f"  • Train Window: {train_df['timestamp'].min()} -> {train_df['timestamp'].max()}")
print(f"  • Val Window:   {val_df['timestamp'].min()} -> {val_df['timestamp'].max()}")
print(f"  • Test Window:  {test_df['timestamp'].min()} -> {test_df['timestamp'].max()}")
assert train_df["timestamp"].max() < val_df["timestamp"].min()
assert val_df["timestamp"].max() < test_df["timestamp"].min()
print("  ✓ Zero forward leakage verified.")

peak_cutoff = float(np.percentile(train_df[target_col], PEAK_PERCENTILE))
print(f"
Operational Peak Threshold (85th percentile of Train): {peak_cutoff:.1f} kWh")

## 4. Model Training & Out-of-Sample Benchmark Evaluation
We train the forecasters and evaluate them against the primary baseline required by the scenario: the **previous day's same-hour consumption** ( = y_{t-24}$).

In [6]:
X_train, y_train = train_df[feature_cols], train_df[target_col]
X_val, y_val = val_df[feature_cols], val_df[target_col]
X_test, y_test = test_df[feature_cols], test_df[target_col]

models = {
    "Baseline (Same Hour Previous Day)": PersistenceBaseline24h(),
    "Baseline (Same Hour Previous Week)": PersistenceBaseline168h(),
    "Ridge Linear Regression": RidgeEnergyForecaster(alpha=12.0),
    "HistGradientBoosting Forecaster": HistGradientBoostingForecaster(),
    "XGBoost Energy Forecaster": XGBoostEnergyForecaster(n_estimators=450, learning_rate=0.03, max_depth=5),
}

for name, m in models.items():
    if name == "XGBoost Energy Forecaster":
        m.fit(X_train, y_train, X_val, y_val)
    else:
        m.fit(X_train, y_train)

df_bench = evaluate_models_comparison(models, X_test, y_test, peak_threshold=peak_cutoff)
df_bench

## 5. 24-Hour Advance Forecast & Operational Peak Advisory
We extract an operational 24-hour test window (Monday, May 19, 2025) featuring elevated ambient temperature and campus attendance.

In [7]:
peak_match = test_df[test_df["timestamp"] == "2025-05-19 00:00:00"].index
start_idx = peak_match[0] if len(peak_match) > 0 else 0
df_24h = test_df.iloc[start_idx : start_idx + 24].copy().reset_index(drop=True)

df_24h["predicted_kwh"] = models["XGBoost Energy Forecaster"].predict(df_24h[feature_cols])
df_24h["baseline_24h_kwh"] = models["Baseline (Same Hour Previous Day)"].predict(df_24h[feature_cols])

engine = CampusAdvisoryEngine(peak_threshold_kwh=peak_cutoff, critical_threshold_kwh=540.0)
advisory = engine.assess_24h_window(df_24h, pred_col="predicted_kwh")

print(f"Projected Total 24h Energy:  {advisory['total_24h_kwh']:,} kWh")
print(f"Projected Peak Demand:       {advisory['peak_demand_kwh']} kWh at {advisory['peak_time']}")
print(f"Flagged Peak Hours ({advisory['n_flagged_hours']} hrs): {', '.join(advisory['flagged_hours'])}")
print(f"Primary Demand Drivers:      {'; '.join(advisory['peak_drivers'])}")

print("
--- RECOMMENDED OPERATIONAL ACTION PLAN ---")
for i, act in enumerate(advisory["action_plan"], 1):
    print(f"{i}. [{act['priority']}] {act['subsystem']}: {act['recommendation']}")
    print(f"   -> {act['technical_details']}")

In [8]:
# Visualizing 24-Hour Forecast vs Actuals and Baseline
fig, ax = plt.subplots(figsize=(12, 5))
times = df_24h["timestamp"]

ax.plot(times, df_24h["electricity_consumption_kwh"], label="Actual Metered Consumption", color="#1a1a1a", lw=2.2, marker="o")
ax.plot(times, df_24h["predicted_kwh"], label="XGBoost 24h Advance Forecast", color="#0b60b0", lw=2.5, marker="s")
ax.plot(times, df_24h["baseline_24h_kwh"], label="Previous Day Persistence Baseline", color="#8c8c8c", lw=1.8, ls="--")
ax.axhline(peak_cutoff, color="#c0392b", ls=":", lw=1.8, label=f"Peak Demand Alert Threshold ({peak_cutoff:.1f} kWh)")

# Shade flagged peak hours
peak_mask = df_24h["predicted_kwh"] >= peak_cutoff
for i in range(len(df_24h)):
    if peak_mask.iloc[i]:
        ax.axvspan(times.iloc[max(0, i-1)], times.iloc[min(len(df_24h)-1, i+1)], color="#c0392b", alpha=0.08)

ax.set_title("24-Hour Campus Electricity Forecast vs. Actuals & Baseline", fontweight="bold")
ax.set_ylabel("Electricity Consumption (kWh)")
ax.set_xlabel("Time of Day")
ax.grid(True, alpha=0.3)
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

## 6. Data Limitations & Operational Boundaries

1. **Weather Forecast Propagation:** In historical training, weather features are exact recorded measurements. In live 24h deployment, the model relies on Numerical Weather Prediction (NWP) forecasts which carry an uncertainty of $\pm 1.5^\circ\text{C}$ to $\pm 3.0^\circ\text{C}$, propagating non-linearly into cooling loads.
2. **Occupancy Proxy Limitations:** Binary occupancy indicates whether spaces are in use, but does not measure localized occupant density (e.g. 200 occupants in a single lecture auditorium produce $\sim 24\text{ kW}$ of body heat concentrating cooling load on one specific air handler).
3. **Thermal Mass Envelope Lag:** Heavy building envelopes store heat; outdoor temperature peaks typically lead indoor chiller demand peaks by 1 to 2 hours.
4. **Operational Boundary:** This prototype is strictly an **Advisory Decision Support System**. It does **not** assert direct digital control or autonomously override Building Automation System (BAS) or SCADA equipment.